Plan -

For detecting the crop type -
On USDA website we are downloading the cropland data layer (CDL) which is a raster(dividing image into cells in a grid) with crop labels. It is 30m resolution which aligns with some sentinel-2 bands.
Each pixel in sentinel-2 imagery (after some processing) would correspond to a CDL pixcel indicating the crop type. The model will learn to predict these labels.

For detecting the field boundaries -
CDL has per-pixel crop labels, but fields are contiguous. It is more straight forward to use vector boundaries from the crop sequence boundaries because they are the outlines of fields.

1. download and alignt sentinel-2 data with USDA data (overlap) - same coordinate system, same spatial resolution, same regions, same time periods. sentinel-2 has bands with 10, 20, 60m resolutions. CDL is 30m. We need to resample sentinel-2 bands to 30m to match CDL.
(
to prevent the model only working for specific region and slow to update, we will use both CDL and vector boundaries as training data,
CDL raster: use it as segmentation mask, each pixel represents a crop type. but for field boundaries use a binary mask (field 1 vs. non-field 0)
crop sequence boundaries: polygons of fields. convert these into raster masks where each field is a separate ploygon with unique ID. This can be used for semantic segamentation(the category of a thing, field in this case) or instance segmentation (the specification of things in category like field1, 2, 3...)
)

2. Training data creation - For each sentinel-2 image, create mask from USDA data. With CDL, mask is crop type per pixel. With vector boundaries, rasterize polygons to create binary mask (field 1 vs. non-field 0) or instance masks.

3. Model training - use supervised learning model like U-Net or similiar CNN for semantic segmentation. Inputs are sentinel-2 bands (all 13 or relevant ones like RGB, NIR, SWIR). Output is the predicted mask (field, non-field)

4. Post processing - convert predicted segmentation mask into vector boundaries with polygonization. Calculate acreage by computing the area of each ploygon in vector data. 

5. Validation - compare predicted boundaries with USDA vector boundaries with metrics like IoU (intersection over union) for semantic segmentation, or precision/recall for boundary detection. for acreage, calculate difference between predicted and ground truth areas.